# Registro histórico: 08_weighted_focal

Origem: `C13` — `Colab Notebooks/Untitled22.ipynb`.

EfficientNet com sampler ponderado e Focal Loss.

**Limitações conhecidas:** Variação metodológica; sem resultado final salvo que permita afirmar superioridade.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from timeit import default_timer as timer
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import precision_recall_curve, average_precision_score

# Configurações
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Seeds para reprodutibilidade
random.seed(91432)
np.random.seed(91432)
torch.manual_seed(91432)
if torch.cuda.is_available():
    torch.cuda.manual_seed(91432)

In [ ]:
# ============================================================================
# 1. CARREGAMENTO E PREPARAÇÃO DOS DADOS
# ============================================================================

DATASET_DIR = "data_ham10000"
metadata_path = os.path.join(DATASET_DIR, "HAM10000_metadata.csv")
df = pd.read_csv(metadata_path)

img_dir_1 = os.path.join(DATASET_DIR, "ham10000_images_part_1")
img_dir_2 = os.path.join(DATASET_DIR, "ham10000_images_part_2")

def get_image_path(image_id):
    filename = image_id + ".jpg"
    path1 = os.path.join(img_dir_1, filename)
    path2 = os.path.join(img_dir_2, filename)
    if os.path.exists(path1):
        return path1
    elif os.path.exists(path2):
        return path2
    return None

df_binary = df.copy()
df_binary["target"] = (df_binary["dx"] == "mel").astype(int)
df_binary["filepath"] = df_binary["image_id"].apply(get_image_path)
df_binary = df_binary.dropna(subset=["filepath"]).reset_index(drop=True)

# Split estratificado
train_df, temp_df = train_test_split(
    df_binary, test_size=0.3, random_state=42, stratify=df_binary["target"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["target"]
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Melanomas - Train: {train_df['target'].sum()} | Val: {val_df['target'].sum()} | Test: {test_df['target'].sum()}")

#

In [ ]:
# ============================================================================
# 2. DATA AUGMENTATION AGRESSIVA PARA MELANOMAS
# ============================================================================

IMG_SIZE = 224
BATCH_SIZE = 16  # Reduzido para permitir mais épocas com mesmo hardware

weights = EfficientNet_B0_Weights.IMAGENET1K_V1
mean = weights.transforms().mean
std = weights.transforms().std

# Augmentation MAIS AGRESSIVA para melanomas
train_transform_mel = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.9, 1.1)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
    transforms.Normalize(mean=mean, std=std),
])

# Augmentation moderada para não-melanomas
train_transform_normal = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

In [ ]:
# ============================================================================
# 3. DATASET CUSTOMIZADO COM AUGMENTATION DIFERENCIADA
# ============================================================================

class HAMDatasetBalanced(Dataset):
    def __init__(self, df, transform_mel=None, transform_normal=None, is_training=True):
        self.df = df.reset_index(drop=True)
        self.transform_mel = transform_mel
        self.transform_normal = transform_normal
        self.is_training = is_training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["filepath"]
        label = row["target"]

        image = Image.open(path).convert("RGB")

        # Aplicar augmentation diferente baseado na classe
        if self.is_training and label == 1 and self.transform_mel:
            image = self.transform_mel(image)
        elif self.is_training and self.transform_normal:
            image = self.transform_normal(image)
        else:
            image = val_test_transform(image)

        label = torch.tensor(label, dtype=torch.float32)
        return image, label


In [ ]:
# ============================================================================
# 4. WEIGHTED SAMPLING PARA BALANCEAR BATCHES
# ============================================================================

def create_weighted_sampler(df):
    """Cria sampler que sobreamostra melanomas"""
    y_train = df["target"].values
    class_counts = np.bincount(y_train)

    # Pesos inversamente proporcionais à frequência
    weights = 1.0 / class_counts[y_train]

    # Aumentar ainda mais o peso dos melanomas
    weights[y_train == 1] *= 2.0

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(weights),
        num_samples=len(weights),
        replacement=True
    )
    return sampler

train_dataset = HAMDatasetBalanced(
    train_df,
    transform_mel=train_transform_mel,
    transform_normal=train_transform_normal,
    is_training=True
)
val_dataset = HAMDatasetBalanced(val_df, is_training=False)
test_dataset = HAMDatasetBalanced(test_df, is_training=False)

# Usar weighted sampler
train_sampler = create_weighted_sampler(train_df)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,  # Usar sampler ao invés de shuffle
    num_workers=2,
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


In [ ]:
# ============================================================================
# 5. FOCAL LOSS - MELHOR QUE BCE PARA DESBALANCEAMENTO
# ============================================================================

class FocalLoss(nn.Module):
    """Focal Loss para lidar com desbalanceamento extremo"""
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            inputs, targets, reduction='none'
        )
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

In [ ]:
# ============================================================================
# 6. MODELO COM DROPOUT MAIS AGRESSIVO
# ============================================================================

base_model = efficientnet_b0(weights=weights)
in_features = base_model.classifier[1].in_features

# Classifier com mais regularização
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.7, inplace=True),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(256, 1)
)

# Descongelar mais camadas para fine-tuning
for param in base_model.features[:-8].parameters():
    param.requires_grad = False

base_model = base_model.to(device)

In [ ]:
# ============================================================================
# 7. LOSS E OPTIMIZER
# ============================================================================

loss_fn = FocalLoss(alpha=0.75, gamma=2.0)

optimizer = optim.AdamW(
    base_model.parameters(),
    lr=5e-5,  # Learning rate menor para fine-tuning
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)

In [ ]:
# ============================================================================
# 8. FUNÇÕES DE TREINO E VALIDAÇÃO COM THRESHOLD AJUSTÁVEL
# ============================================================================

def train_step(model, data_loader, loss_fn, optimizer, device):
    model.train()
    train_loss, train_acc = 0, 0
    all_probs, all_labels = [], []

    for X, y in tqdm(data_loader, desc="Training", leave=False):
        X, y = X.to(device), y.to(device)

        y_pred_logits = model(X).squeeze(1)
        loss = loss_fn(y_pred_logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred_probs = torch.sigmoid(y_pred_logits)
        y_pred_class = (y_pred_probs >= 0.5).float()

        acc = (y_pred_class == y).float().mean().item() * 100

        train_loss += loss.item()
        train_acc += acc

        all_probs.extend(y_pred_probs.detach().cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    train_loss /= len(data_loader)
    train_acc /= len(data_loader)

    return train_loss, train_acc

def val_step(model, data_loader, loss_fn, device, threshold=0.5):
    model.eval()
    val_loss, val_acc = 0, 0
    all_probs, all_labels = [], []

    with torch.inference_mode():
        for X, y in tqdm(data_loader, desc="Validation", leave=False):
            X, y = X.to(device), y.to(device)

            y_pred_logits = model(X).squeeze(1)
            loss = loss_fn(y_pred_logits, y)

            y_pred_probs = torch.sigmoid(y_pred_logits)
            y_pred_class = (y_pred_probs >= threshold).float()

            acc = (y_pred_class == y).float().mean().item() * 100

            val_loss += loss.item()
            val_acc += acc

            all_probs.extend(y_pred_probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    val_loss /= len(data_loader)
    val_acc /= len(data_loader)

    # Calcular F1 para melanoma
    y_pred = (np.array(all_probs) >= threshold).astype(int)
    y_true = np.array(all_labels)

    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_melanoma = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return val_loss, val_acc, f1_melanoma, all_probs, all_labels

In [ ]:
# ============================================================================
# 9. TREINAMENTO COM MONITORAMENTO DE F1-SCORE
# ============================================================================

EPOCHS = 60
PATIENCE = 10
MIN_DELTA = 0.001

best_f1 = 0.0
best_model_state = None
epochs_no_improve = 0

train_losses, train_accs = [], []
val_losses, val_accs, val_f1s = [], [], []

print("\n" + "="*60)
print("INICIANDO TREINAMENTO FOCADO EM MELANOMA")
print("="*60)

train_start_time = timer()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 40)

    train_loss, train_acc = train_step(
        model=base_model,
        data_loader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device
    )

    val_loss, val_acc, val_f1, val_probs, val_labels = val_step(
        model=base_model,
        data_loader=val_dataloader,
        loss_fn=loss_fn,
        device=device,
        threshold=0.5
    )

    # Scheduler baseado em F1 ao invés de loss
    scheduler.step(val_f1)

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_f1s.append(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"Val F1-Melanoma: {val_f1:.4f}")
    print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Early stopping baseado em F1
    if val_f1 > best_f1 + MIN_DELTA:
        best_f1 = val_f1
        best_model_state = base_model.state_dict().copy()
        epochs_no_improve = 0
        print(f"🎯 Novo melhor F1: {best_f1:.4f}")
    else:
        epochs_no_improve += 1
        print(f"⏳ Sem melhoria: {epochs_no_improve}/{PATIENCE}")

        if epochs_no_improve >= PATIENCE:
            print(f"🛑 Early stopping em epoch {epoch+1}")
            break

train_end_time = timer()
total_time = train_end_time - train_start_time

print(f"\n✅ Treinamento concluído em {total_time:.2f}s")
print(f"Melhor F1-Score (Melanoma): {best_f1:.4f}")

In [ ]:
# ============================================================================
# 10. ENCONTRAR THRESHOLD ÓTIMO
# ============================================================================

if best_model_state:
    base_model.load_state_dict(best_model_state)

print("\n" + "="*60)
print("ENCONTRANDO THRESHOLD ÓTIMO PARA MAXIMIZAR RECALL")
print("="*60)

# Coletar todas as probabilidades do conjunto de validação
base_model.eval()
all_probs, all_labels = [], []

with torch.inference_mode():
    for X, y in val_dataloader:
        X = X.to(device)
        logits = base_model(X).squeeze(1)
        probs = torch.sigmoid(logits)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Testar diferentes thresholds
thresholds = np.arange(0.1, 0.9, 0.05)
best_threshold = 0.5
best_score = 0

results = []
for thresh in thresholds:
    y_pred = (all_probs >= thresh).astype(int)

    tp = ((y_pred == 1) & (all_labels == 1)).sum()
    fp = ((y_pred == 1) & (all_labels == 0)).sum()
    fn = ((y_pred == 0) & (all_labels == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results.append({
        'threshold': thresh,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

    # Priorizar recall (queremos capturar melanomas)
    score = 0.3 * precision + 0.7 * recall
    if score > best_score:
        best_score = score
        best_threshold = thresh

results_df = pd.DataFrame(results)
print(f"\n🎯 Threshold ótimo: {best_threshold:.2f}")
print(f"Maximiza: 0.3*Precision + 0.7*Recall = {best_score:.4f}")

# Visualizar curva threshold
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(results_df['threshold'], results_df['precision'], 'b-', label='Precision')
plt.plot(results_df['threshold'], results_df['recall'], 'r-', label='Recall')
plt.axvline(best_threshold, color='g', linestyle='--', label=f'Best={best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision vs Recall')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(results_df['threshold'], results_df['f1'], 'purple')
plt.axvline(best_threshold, color='g', linestyle='--')
plt.xlabel('Threshold')
plt.ylabel('F1-Score')
plt.title('F1-Score por Threshold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = roc_auc_score(all_labels, all_probs)
plt.plot(fpr, tpr, label=f'ROC (AUC={roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# 11. AVALIAÇÃO FINAL NO TEST SET COM THRESHOLD OTIMIZADO
# ============================================================================

print("\n" + "="*60)
print("AVALIAÇÃO FINAL NO TEST SET")
print("="*60)

base_model.eval()
all_probs, all_labels = [], []

with torch.inference_mode():
    for X, y in tqdm(test_dataloader, desc="Testing"):
        X = X.to(device)
        logits = base_model(X).squeeze(1)
        probs = torch.sigmoid(logits)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Predições com threshold otimizado
y_pred_optimized = (all_probs >= best_threshold).astype(int)

print(f"\n📊 Resultados com Threshold Otimizado ({best_threshold:.2f}):")
print("\n" + classification_report(
    all_labels,
    y_pred_optimized,
    target_names=['Non-Melanoma', 'Melanoma'],
    digits=3
))

# Matriz de confusão
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
cm = confusion_matrix(all_labels, y_pred_optimized)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Mel', 'Mel'],
            yticklabels=['Non-Mel', 'Mel'])
plt.title(f'Confusion Matrix (threshold={best_threshold:.2f})')
plt.ylabel('True')
plt.xlabel('Predicted')

# Comparação com threshold 0.5
y_pred_default = (all_probs >= 0.5).astype(int)
plt.subplot(1, 2, 2)
cm_default = confusion_matrix(all_labels, y_pred_default)
sns.heatmap(cm_default, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Non-Mel', 'Mel'],
            yticklabels=['Non-Mel', 'Mel'])
plt.title('Confusion Matrix (threshold=0.5)')
plt.ylabel('True')
plt.xlabel('Predicted')

plt.tight_layout()
plt.show()

# Métricas específicas para melanoma
tp = cm[1, 1]
fp = cm[0, 1]
fn = cm[1, 0]
tn = cm[0, 0]

sensitivity = tp / (tp + fn)  # Recall para melanoma
specificity = tn / (tn + fp)
ppv = tp / (tp + fp)  # Precision para melanoma
npv = tn / (tn + fn)

print("\n📈 Métricas Clínicas:")
print(f"Sensibilidade (Recall Melanoma): {sensitivity:.3f} - {tp}/{tp+fn} melanomas detectados")
print(f"Especificidade: {specificity:.3f}")
print(f"Valor Preditivo Positivo (Precision): {ppv:.3f}")
print(f"Valor Preditivo Negativo: {npv:.3f}")
print(f"ROC-AUC: {roc_auc_score(all_labels, all_probs):.3f}")

In [ ]:
# ============================================================================
# 12. SALVAR MODELO
# ============================================================================

MODEL_PATH = Path("/content/models")
MODEL_PATH.mkdir(exist_ok=True)

model_save_path = MODEL_PATH / "efficientnet_melanoma_optimized.pth"
torch.save({
    'model_state_dict': base_model.state_dict(),
    'threshold': best_threshold,
    'f1_score': best_f1,
    'metrics': {
        'sensitivity': sensitivity,
        'specificity': specificity,
        'roc_auc': roc_auc_score(all_labels, all_probs)
    }
}, model_save_path)

print(f"\n✅ Modelo salvo em: {model_save_path}")
print("\n" + "="*60)
print("RESUMO FINAL")
print("="*60)
print(f"Threshold ótimo: {best_threshold:.2f}")
print(f"Sensibilidade (detectar melanomas): {sensitivity:.1%}")
print(f"Especificidade: {specificity:.1%}")
print(f"F1-Score Melanoma: {best_f1:.3f}")
print(f"ROC-AUC: {roc_auc_score(all_labels, all_probs):.3f}")
print("="*60)